In [9]:
# Prepare Bloomberg security identifiers for bankrupt and nonbankrupt firms.
# Use all nonbankrupt firms and collect their history across the bankrupt firms' date range.

In [10]:
import sys as _sys
from pathlib import Path as _Path
_UTILS_ROOT = next(path for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
                   if (path / "notebook_utils.py").is_file())
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

BANKRUPT_PATH = ROOT / "data/raw/bankrupt/data.csv"
NONBANKRUPT_PATH = ROOT / "data/raw/nonbankrupt/data.csv"
OUTPUT_DIR = project_paths(ROOT).intermediate
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOOKBACK_YEARS = 5


In [11]:
bankrupt = pd.read_csv(BANKRUPT_PATH, dtype={"ticker": "string", "name": "string", "bloomberg_identifier": "string"})
required = {"bankruptcy_date", "ticker", "name", "bloomberg_identifier"}
missing = required - set(bankrupt.columns)
if missing:
    raise ValueError(f"Bankrupt CSV is missing columns: {sorted(missing)}")
bankrupt["bankruptcy_date"] = pd.to_datetime(bankrupt["bankruptcy_date"], errors="coerce")
bankrupt["ticker"] = bankrupt["ticker"].str.strip().str.upper()
bankrupt["name"] = bankrupt["name"].str.strip()
bankrupt["bloomberg_identifier"] = bankrupt["bloomberg_identifier"].fillna("").str.strip()
bankrupt = bankrupt.loc[bankrupt["bloomberg_identifier"].ne("")].copy()
if bankrupt.empty:
    raise ValueError("No bankrupt records have a nonempty Bloomberg identifier.")
if bankrupt["bankruptcy_date"].isna().any():
    raise ValueError("Bankrupt records with Bloomberg identifiers must have valid filing dates.")
bankrupt["company_id"] = "SA_" + bankrupt.index.astype(str)
bankrupt["prediction_date"] = bankrupt["bankruptcy_date"] - pd.Timedelta(days=90)
bankrupt["start_date"] = bankrupt["prediction_date"] - pd.DateOffset(years=LOOKBACK_YEARS)
bankrupt["end_date"] = bankrupt["prediction_date"]
bankrupt = bankrupt.sort_values(["bankruptcy_date", "company_id"]).copy()

# Download enough history for any later control match to a bankrupt event.
HISTORICAL_START_DATE = bankrupt["start_date"].min()
HISTORICAL_END_DATE = bankrupt["end_date"].max()
display(bankrupt)


,bankruptcy_date,ticker,name,bloomberg_identifier,company_id,prediction_date,start_date,end_date
222,1998-09-16,SNTVQ,"Sun Television and Appliances, Inc.",SNTVQ US Equity,SA_222,1998-06-18,1993-06-18,1998-06-18
187,1998-10-05,BOST,"Boston Chicken, Inc.",BCHXQ US Equity,SA_187,1998-07-07,1993-07-07,1998-07-07
135,1998-12-27,JSIBQ,"JumboSports, Inc.",JSIBQ US Equity,SA_135,1998-09-28,1993-09-28,1998-09-28
4,1998-12-29,SLNT,Salant Corp. (1998),2777116Q US Equity,SA_4,1998-09-30,1993-09-30,1998-09-30
26,1999-02-11,WIRL,"Wireless One, Inc.",WIRLQ US Equity,SA_26,1998-11-13,1993-11-13,1998-11-13
...,...,...,...,...,...,...,...,...
146,2026-06-22,SNBRQ,Sleep Number Corp,SNBRQ US Equity,SA_146,2026-03-24,2021-03-24,2026-03-24
301,2026-06-23,DEVSF,Devvstream Corp,DEVSF US Equity,SA_301,2026-03-25,2021-03-25,2026-03-25
63,2026-06-24,ADTX,Aditxt Inc,ADTX US Equity,SA_63,2026-03-26,2021-03-26,2026-03-26
148,2026-07-01,ORGN,Origin Materials Inc,ORGN US Equity,SA_148,2026-04-02,2021-04-02,2026-04-02


In [12]:
bankrupt_output = bankrupt[["company_id", "bankruptcy_date", "ticker", "name", "bloomberg_identifier", "prediction_date", "start_date", "end_date"]].copy()


In [13]:
nonbankrupt = pd.read_csv(NONBANKRUPT_PATH, dtype={"ticker": "string", "name": "string"})
required_nonbankrupt = {"ticker", "name", "Sector", "Industry"}
missing_nonbankrupt = required_nonbankrupt - set(nonbankrupt.columns)
if missing_nonbankrupt:
    raise ValueError(f"Non-bankrupt CSV is missing columns: {sorted(missing_nonbankrupt)}")
nonbankrupt_input_count = len(nonbankrupt)
if nonbankrupt.empty:
    raise ValueError("The non-bankrupt CSV is empty.")
nonbankrupt["ticker"] = nonbankrupt["ticker"].str.strip().str.upper()
nonbankrupt["name"] = nonbankrupt["name"].str.strip()
invalid_tickers = nonbankrupt["ticker"].fillna("").eq("")
if invalid_tickers.any():
    raise ValueError(f"Missing non-bankrupt tickers at CSV rows {(nonbankrupt.index[invalid_tickers] + 2).tolist()}; correct them so every company can be included.")
nonbankrupt["bloomberg_identifier"] = nonbankrupt["ticker"] + " US Equity"
nonbankrupt["company_id"] = "NB_" + nonbankrupt.index.astype(str)
for column in ["Sector", "Industry"]:
    nonbankrupt[column] = nonbankrupt[column].astype("string").fillna("Unknown").str.strip().replace("", "Unknown")

# Retain all input rows, including their original company IDs and ordering.
# These are collection bounds, not a matched prediction date. After fetching
# historical data, assign a pseudo-date and restrict features to earlier data.
nonbankrupt["prediction_date"] = pd.NaT
nonbankrupt["start_date"] = HISTORICAL_START_DATE
nonbankrupt["end_date"] = HISTORICAL_END_DATE
nonbankrupt_output = nonbankrupt[["company_id", "ticker", "name", "Sector", "Industry", "bloomberg_identifier", "prediction_date", "start_date", "end_date"]].copy()

display(nonbankrupt_output.head())


,company_id,ticker,name,Sector,Industry,bloomberg_identifier,prediction_date,start_date,end_date
0,NB_0,A,Agilent Technologies Inc. Common Stock,Industrials,Biotechnology: Laboratory Analytical Instruments,A US Equity,NaT,1993-06-18,2026-05-02
1,NB_1,AA,Alcoa Corporation Common Stock,Industrials,Aluminum,AA US Equity,NaT,1993-06-18,2026-05-02
2,NB_2,AAL,American Airlines Group Inc. Common Stock,Consumer Discretionary,Air Freight/Delivery Services,AAL US Equity,NaT,1993-06-18,2026-05-02
3,NB_3,AAOI,Applied Optoelectronics Inc. Common Stock,Technology,Semiconductors,AAOI US Equity,NaT,1993-06-18,2026-05-02
4,NB_4,AAON,AAON Inc. Common Stock,Industrials,Industrial Machinery/Components,AAON US Equity,NaT,1993-06-18,2026-05-02


In [14]:
bankrupt_list = bankrupt_output.copy()
bankrupt_list["company_type"] = "bankrupt"
nonbankrupt_list = nonbankrupt_output.copy()
nonbankrupt_list["company_type"] = "nonbankrupt"
bloomberg_input = pd.concat([bankrupt_list, nonbankrupt_list], ignore_index=True)
bloomberg_input = bloomberg_input[["company_id", "company_type", "bankruptcy_date", "ticker", "name", "Sector", "Industry", "bloomberg_identifier", "prediction_date", "start_date", "end_date"]]

display(bloomberg_input.head(20))


,company_id,company_type,bankruptcy_date,ticker,name,Sector,Industry,bloomberg_identifier,prediction_date,start_date,end_date
0,SA_222,bankrupt,1998-09-16,SNTVQ,"Sun Television and Appliances, Inc.",<NA>,<NA>,SNTVQ US Equity,1998-06-18,1993-06-18,1998-06-18
1,SA_187,bankrupt,1998-10-05,BOST,"Boston Chicken, Inc.",<NA>,<NA>,BCHXQ US Equity,1998-07-07,1993-07-07,1998-07-07
2,SA_135,bankrupt,1998-12-27,JSIBQ,"JumboSports, Inc.",<NA>,<NA>,JSIBQ US Equity,1998-09-28,1993-09-28,1998-09-28
3,SA_4,bankrupt,1998-12-29,SLNT,Salant Corp. (1998),<NA>,<NA>,2777116Q US Equity,1998-09-30,1993-09-30,1998-09-30
4,SA_26,bankrupt,1999-02-11,WIRL,"Wireless One, Inc.",<NA>,<NA>,WIRLQ US Equity,1998-11-13,1993-11-13,1998-11-13
5,SA_172,bankrupt,1999-03-01,PNFTQ,Penn Traffic Co (1999),<NA>,<NA>,PNFTQ US Equity,1998-12-01,1993-12-01,1998-12-01
6,SA_80,bankrupt,1999-03-15,SME,"Service Merchandise Company, Inc.",<NA>,<NA>,SVCDQ US Equity,1998-12-15,1993-12-15,1998-12-15
7,SA_134,bankrupt,1999-09-03,COSE,"Costilla Energy, Inc.",<NA>,<NA>,COSEQ US Equity,1999-06-05,1994-06-05,1999-06-05
8,SA_52,bankrupt,1999-09-16,TRSM,"Trism, Inc.",<NA>,<NA>,2471671D US Equity,1999-06-18,1994-06-18,1999-06-18
9,SA_145,bankrupt,1999-11-04,FEET,"Just For Feet, Inc.",<NA>,<NA>,FEETQ US Equity,1999-08-06,1994-08-06,1999-08-06


In [15]:

output_path = OUTPUT_DIR / "bloomberg_security_list.csv"
bloomberg_input.to_csv(output_path, index=False, date_format="%Y-%m-%d")